In [1]:
#준비 셀 1. 학습용 50,000장과 검증용 10,000장, 로더 두 개
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
device = "cuda" if torch.cuda.is_available() else "cpu"
train_all = datasets.MNIST(root="data", train=True, download=True,
 transform=transforms.ToTensor())
g = torch.Generator().manual_seed(0) # 2주차와 같은 분리
train_set, val_set = random_split(train_all, [50000, 10000], generator=g)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False)
print(len(train_loader), len(val_loader)) 

100.0%
100.0%
100.0%
100.0%

782 40


In [ ]:
#준비 셀 2.3주차 MLP에서 두 곳이 바뀐다
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flat = nn.Flatten() # 바뀐 곳 1 (N, 1, 28, 28) → (N, 784)
        self.sc1 = nn.Linear(784, 100)
        self.act = nn.ReLU()
        self.sc2 = nn.Linear(100, 10)
    def forward(self, x):
        x = self.flat(x) # 바뀐 곳 2, 먼저 편다
        return self.sc2(self.act(self.sc1(x)))
    
loss_fn = nn.CrossEntropyLoss()
print(MLP()(torch.zeros(2,1,28,28)).shape) # → torch.Size([2, 10])

torch.Size([2, 10])


In [3]:
#예제 4-1. 손으로 한 계산과 nn.CrossEntropyLoss 비교

import torch
import torch.nn as nn
logits = torch.tensor([[2.0, 1.0,0.1]]) #샘플 1개, 클래스 3개의 점수
target = torch.tensor([0]) #정답 클래스 0
p = torch.softmax(logits, dim = 1) # 점수 → 확률
print(p) # → tensor([[0.6590, 0.2424, 0.0986]])
print(-torch.log(p[0,0])) # → tensor(0.4170) # 정답 클래스 0의 확률에 대한 -log
loss_fn = nn.CrossEntropyLoss()
print(loss_fn(logits, target)) # → tensor(0.4170) 같은 값, softmax와 log를 합쳐서 계산

tensor([[0.6590, 0.2424, 0.0986]])
tensor(0.4170)
tensor(0.4170)


In [5]:
#예제 4-2. 학습 전 모델의 확률과 첫 loss
torch.manual_seed(0)
model = MLP().to(device)
x, y = next(iter(train_loader)) # 학습용 데이터에서 배치 1개
x, y = x.to(device), y.to(device)
p = torch.softmax(model(x), dim=1) # 모델의 점수 → 확률
print(round(p.min().item(), 2), round(p.max().item(), 2)) 
# → (예) 0.08 0.12

print(round(loss_fn(model(x), y).item(), 4)) # → (예) 2.2942 첫loss
print(round(-torch.log(torch.tensor(0.1)).item(), 4)) # → 2.3026 (-log(1/10))

0.08 0.12
2.2942
2.3026


In [6]:
#예제 4-3. 맞힌 개수와 loss를 함께 보기
import torch
import torch.nn as nn
loss_fn = nn.CrossEntropyLoss()
labels = torch.tensor([0, 1, 2, 0]) # 샘플 4개의 정답
out_a = torch.tensor([[4.0, 0.0, 0.0], [0.0, 4.0, 0.0], # 모델 A, 확신하며 맞힌다
 [0.0, 0.0, 4.0], [0.0, 2.0, 0.0]])
out_b = torch.tensor([[1.0, 0.8, 0.0], [0.8, 1.0, 0.0], # 모델 B, 겨우 맞힌다
 [0.0, 0.8, 1.0], [0.0, 2.0, 0.0]])
print((out_a.argmax(dim = 1)==labels).sum().item(), (out_b.argmax(dim = 1) == labels).sum().item()) 
# → 3 3 맞힌 개수는 같다
print(round(loss_fn(out_a, labels).item(), 4), round(loss_fn(out_b, labels).item(), 4))
# → 0.5869 1.1467 (Loss는 B가 약 2배)

3 3
0.5869 1.1467


In [7]:
#실습 
#1. 정답 확률이 0.9, 0.5, 0.1일 때의 loss를 -torch.log(...)로 계산
#2. 예제 4-1의 점수에서 정답 점수 2.0을 4.0으로 바꾸면 loss가 커질지 작아질지 예측한 뒤 확인하기
#3. 원래 점수 [2.0, 1.0, 0.1]에서 정답만 2번 클래스로 바꾸면 loss는?
import torch
import torch.nn as nn
print(-torch.log(torch.tensor([0.9, 0.5, 0.1]))) # 1 정답 확률별 loss
loss_fn = nn.CrossEntropyLoss()
logits = torch.tensor([[4.0, 1.0, 0.1]]) # 2 정답 점수 4.0으로 바꾸기
print(loss_fn(logits, torch.tensor([0]))) # → 0.0486 loss가 작아진다
logits = torch.tensor([[2.0, 1.0, 0.1]]) # 3 정답만 2번 클래스로 바꾸기
print(loss_fn(logits, torch.tensor([2]))) # → 2.3026 loss




tensor([0.1054, 0.6931, 2.3026])
tensor(0.0677)
tensor(2.3170)


In [8]:
#실행해서 검증
import torch
import torch.nn as nn
print(-torch.log(torch.tensor(1 / 100))) # 1 클래스 100개
# → tensor(4.6052)
L = -torch.log(torch.tensor([0.1, 0.2, 0.5, 0.6]))
print(round((L[0] - L[1]).item(), 4)) # 2 0.1 → 0.2
# → 0.6931
print(round((L[2] - L[3]).item(), 4)) # 2 0.5 → 0.6
# → 0.1823
out = torch.zeros(1, 10) # 3 softmax를 거친 출력
out[0, 0] = 1.0 # 정답 확률 1, 완벽한 경우
print(nn.CrossEntropyLoss()(out, torch.tensor([0])))
# → tensor(1.4612) 완벽해도 1.46 아래로 못 내려간다

tensor(4.6052)
0.6931
0.1823
tensor(1.4612)


In [9]:
#예제 4-4. 앞 장의 계산을 코드로
import torch 
x, t = torch.tensor(2.0), torch.tensor(5.0) #입력 x와 정답 t
w = torch.tensor (3.0, requires_grad=True) # 파라미터 두개 
b = torch.tensor(1.0, requires_grad = True)
pred = w * x + b # forward, 예측
loss = (pred - t) ** 2 # 손실 함수
loss.backward() # backward
print(pred.item(), loss.item()) # → 7.0 4.0
print(w.grad, b.grad) # → tensor(8.) tensor(4.) 손으로 구한 8과 4
with torch.no_grad(): # 기울기 반대 방향으로 한 걸음
 w -= 0.05 * w.grad
 b -= 0.05 * b.grad
print(round(((w * x + b - t) ** 2).item(), 4)) # → 1.0 (loss가 4에서 1로)

7.0 4.0
tensor(8.) tensor(4.)
1.0


In [10]:
#예제 4-5. Cross Entropy의 backward가 logits에 남긴 값
import torch
import torch.nn as nn
loss_fn = nn.CrossEntropyLoss()
logits = torch.tensor([[2.0, 1.0, 0.1]], requires_grad=True)
loss = loss_fn(logits, torch.tensor([0]))
loss.backward()
print(torch.softmax(logits.detach(),dim = 1)) # → tensor([[0.6590, 0.2424, 0.0986]]) 
print(logits.grad) # → tensor([[-0.3410, 0.2424, 0.0986]]) 정답 클래스는 음수, 나머지는 양수

tensor([[0.6590, 0.2424, 0.0986]])
tensor([[-0.3410,  0.2424,  0.0986]])


In [14]:
#예제 4-6. 같은 seed로 예제 4-2의 모델과 배치를 다시 만들고 시작

import torch
import torch.nn as nn


class MLP(nn.Module):

  def __init__(self):
    super().__init__()
    # 예제 4-6의 weight.grad shape([100, 784])를 역산하면 입력 784차원, 출력 100차원입니다.
    self.fc1 = nn.Linear(784, 100)
    self.fc2 = nn.Linear(100, 10)  # bias.grad shape([10])인 것으로 보아 최종 출력은 10차원

  def forward(self, x):
    # 예제 4-2 / 4-6의 흐름에 맞는 순전파 (예: ReLU 활성화 함수 포함 가능)
    x = x.view(x.size(0), -1)  # 이미지를 1차원으로 펼치는 경우
    x = torch.relu(self.fc1(x))
    x = self.fc2(x)
    return x

torch.manual_seed(0) # 예제 4-2와 같은 모델, 같은 배치
model = MLP().to(device)
x, y = next(iter(train_loader))
x, y = x.to(device), y.to(device)
loss = loss_fn(model(x), y)
print(model.fc1.weight.grad) # → None (아직 없음)
loss.backward() # 역전파 한 줄
print(model.fc1.weight.grad.shape) # → torch.Size([100, 784])
print(model.fc2.bias.grad.shape) # → torch.Size([10])
print(sum(p.grad.numel() for p in model.parameters())) # → 79510

None
torch.Size([100, 784])
torch.Size([10])
79510


In [15]:
#예제 4-7. 예제 4-6에 이어서 실행
opt = torch.optim.SGD(model.parameters(), lr=0.1) # 갱신 규칙 SGD
b0 = model.fc2.bias[0].item() # 파라미터 하나의 갱신 전 값
g0 = model.fc2.bias.grad[0].item() # 그 파라미터의 기울기
opt.step() # 79,510개를 한 번에 갱신
print(round(b0 - 0.1 * g0, 4)) # → (예) 0.0894 손으로 계산한 값
print(round(model.fc2.bias[0].item(), 4)) # → (예) 0.0894 (step 후 실제 값)
print(round(loss.item(), 4), round(loss_fn(model(x), y).item(), 4))
# → (예) 2.2942 2.2643 같은 배치의 loss가 줄었다

0.0894
0.0894
2.2942 2.2643


In [16]:
#예제 4-8. 예제 4-7에 이어서 실행
opt.zero_grad() # 기울기 초기화
loss = loss_fn(model(x), y).backward() #기울기 계산
g1 = model.fc2.bias.grad[0].item() # 갱신 후 기울기
loss_fn(model(x),y).backward() # 기울기 계산
g2 = model.fc2.bias.grad[0].item() # 두 번째 기울기
print(round(g2/g1, 4)) # → (예) 2.0 두 번째 기울기가 첫 번째의 2배

2.0


In [34]:
#1. 예제 4-5에서 정답을 2번 클래스로 바꾸면 세 기울기의 부호가 어떻게 될지 예측한 뒤 확인하기
#2. torch.optim.SGD([logits], lr=1.0)로 opt를 만들어 한 걸음 갱신하고 loss가 줄었는지 확인하기
#3. 네 줄을 10번 반복한 뒤 확률을 출력해 정답 확률 보기

import torch
import torch.nn as nn

loss_fn = nn.CrossEntropyLoss()
target = torch.tensor([2])  # 정답은 2번
logits = torch.tensor([[2.0, 1.0, 0.1]], requires_grad=True)

# 1. 초기 상태 확인
loss = loss_fn(logits, target)
loss.backward()
print(torch.softmax(logits.detach(), dim=1)) 
print(logits.grad)

opt = torch.optim.SGD([logits], lr=1.0)

# 2. 한 걸음 갱신 (backward 전 loss를 다시 계산해야 함)
opt.zero_grad()
loss = loss_fn(logits, target)  # 새로운 그래프 생성
loss.backward()
opt.step()                      # 한 걸음 갱신

# 갱신 성공 여부 확인 방법: loss 값이 줄었는지 확인
loss_after = loss_fn(logits, target).item()
print(round(loss.item(), 4), round(loss_after, 4))

# 3. 10번 반복 학습
for step in range(10):
    opt.zero_grad()
    loss = loss_fn(logits, target)  # 매 반복마다 새로운 계산 그래프 생성
    loss.backward()
    opt.step()

# 최종 정답 확률 확인
print(torch.softmax(logits.detach(), dim=1))



tensor([[0.6590, 0.2424, 0.0986]])
tensor([[ 0.6590,  0.2424, -0.9014]])
2.317 1.1594
tensor([[0.0361, 0.0305, 0.9334]])


In [30]:
#실습 
#1. 배치 하나를 고정하고 네 줄을 100번 반복하며 20 step마다 loss 출력하기
#2. 그 배치 64개 중 몇 개를 맞히는지 확인하기
#3. 검증 데이터에서 배치를 하나 꺼내 몇 개를 맞히는지 비교하기

torch.manual_seed(0)
model = MLP().to(device)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
x, y = next(iter(train_loader))
x, y = x.to(device), y.to(device)
for step in range(100):
    loss = loss_fn(model(x), y)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 20 == 0:
        print(step, round(loss.item(), 4))
print((model(x).argmax(dim =1) == y).sum().item()) # → (예) 64 배치 64개 모두 맞힘
x, y = next(iter(val_loader))
x, y = x.to(device), y.to(device)
print((model(x).argmax(dim = 1) == y).sum().item(), len(y)) # → (예) 256 검증 배치 256개 모두 맞힘



0 2.2942
20 1.5677
40 0.7659
60 0.373
80 0.209
64
165 256


In [ ]:
#문제 
#1. 학습 루프에서 opt.step()을 빠뜨리면 어떻게 될까? 오류가 날까?
#2. opt.zero_grad()를 빠뜨리고 1 epoch을 학습하면 검증 정확도는?
#3. 실수로 with torch.no_grad(): 블록 안에서 loss를 만들고 loss.backward()를 부르면?

#답
#1. 오류는 나지 않지만, 모델의 파라미터가 갱신되지 않아 학습이 진행되지 않습니다. 따라서 loss 값이 줄어들지 않고, 검증 정확도도 향상되지 않습니다.
#2. opt.zero_grad()를 빠뜨리면, 이전 배치의 기울기가 누적되어 모델의 파라미터가 잘못 갱신될 수 있습니다. 이로 인해 학습이 불안정해지고, 검증 정확도가 낮아질 수 있습니다.
#3. with torch.no_grad(): 블록 안에서는 기울기 계산이 비활성화되므로, loss.backward()를 호출해도 기울기가 계산되지 않습니다. 따라서 모델의 파라미터가 갱신되지 않고, 학습이 진행되지 않습니다.



In [ ]:
#예제 4-9. 검증 함수 (표준 틀)
def evaluate(model, loader):  # 검증 함수, 표준 틀이라 복사해서 쓴다
    model.eval()  # 평가 모드로 전환
    total_loss, correct = 0.0, 0
    with torch.no_grad():  # 기울기 계산 끄기
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            total_loss += loss_fn(out, y).item() * len(y)  # 평균 × 개수
            correct += (out.argmax(dim=1) == y).sum().item()
            
    n = len(loader.dataset)  # 전체 샘플 수 (직접 쓰지 않는다)
    return total_loss / n, correct / n

In [37]:
#점검 셀, 학습 전 모델로 확인
torch.manual_seed(0)
vl, va = evaluate(MLP().to(device), val_loader) # 학습 전 모델로 점검
print(round(vl, 4), round(va, 2)) # → (예) 2.3024 8.74 학습 전 수준이 나와야 한다

2.3024 0.09


In [40]:
#예제 4-10. 학습 함수 (표준 틀)
def train_one_epoch(model, loader, opt): # 학습 함수, 1 epoch을 돈다

    model.train()
    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item() * len(y)

    return total_loss / len(loader.dataset)